In [ ]:
%%time

# pretrain後のfintuningように修正中
# PyTorch版 HFViT
# version 1.0.0 20250508 とりあえず動くようになった。かなり素。
# version 1.0.1 20250516 pretrain後のfitunignように修正中
#

# # ---------------------------------------------------------
# 0.  ライブラリのインストール（まだなら）
# ---------------------------------------------------------
# pip install torch torchvision transformers datasets evaluate
# pip install timm  # CutMix用

WHERE_AM_I = 'Google Colab' #'Local Linux' #'Google Colab'
if WHERE_AM_I == 'Google Colab':
    from google.colab import drive
    drive.mount('/content/drive')

    !pip install evaluate

import os, torch, PIL
if WHERE_AM_I == 'Google Colab':
    os.environ["WANDB_DISABLED"] = "true"
from torchvision import datasets, transforms
from torchvision.transforms import RandAugment, RandomErasing
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoImageProcessor,
                          AutoModelForImageClassification,
                          TrainingArguments, Trainer, TrainerCallback,
                          ViTMAEForPreTraining, ViTForImageClassification, AutoConfig)
import evaluate
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             roc_curve, auc, precision_recall_curve)
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import datetime
import timm
from timm.data.mixup import Mixup
from torch.optim import Optimizer
from typing import Optional, Dict, List, Union, Tuple
import sys
if WHERE_AM_I == 'Google Colab':
    sys.path.append('/content/drive/MyDrive/takotsubo_ami_ai_ecg')
import ek_schedule_exe
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from scipy.special import softmax
import tensorboard
from collections import OrderedDict
from collections import Counter
import torch.nn as nn
import torch.nn.functional as F
import dataclasses as dc

# ---------------------------------------------------------
# 1. 定数・ラベル定義
# ---------------------------------------------------------

NOTEBOOK_STEM = "tako_ami_ft2_vit_prod_gd_20250817_5"
#どこに保存してある MAE の重みを使うか
#FT3_CKPT_PARENT_DIR = "/path/to/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_20250521_2_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/path/to/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_20250607_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/content/drive/MyDrive/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_gd_20250730_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/path/to/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_20250801_2_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/content/drive/MyDrive/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_gd_20250804_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/content/drive/MyDrive/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_gd_20250810_3_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/content/drive/MyDrive/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_gd_20250806_prt_ft_3class_ckpt"
#FT3_CKPT_PARENT_DIR = "/content/drive/MyDrive/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_gd_20250806_2_prt_ft_3class_ckpt"
FT3_CKPT_PARENT_DIR = "/content/drive/MyDrive/checkpoints/ft_3_class/vit/tako_ami_ft3_vit_prod_gd_20250814_4_prt_ft_3class_ckpt"

#FT3_CKPT_DIR = f"{FT3_CKPT_PARENT_DIR}/checkpoint-6728"
FT3_CKPT_DIR = f"{FT3_CKPT_PARENT_DIR}/checkpoint-9251"
#FT3_CKPT_DIR = f"{FT3_CKPT_PARENT_DIR}/checkpoint-8410"
#FT3_CKPT_DIR = f"{FT3_CKPT_PARENT_DIR}/checkpoint-13456"
#FT3_CKPT_DIR = f"{FT3_CKPT_PARENT_DIR}/checkpoint-36500"
#FT3_CKPT_DIR = f"{FT3_CKPT_PARENT_DIR}/checkpoint-6570"

if WHERE_AM_I == 'Google Colab':
    SAVE_DIR = f"/content/drive/MyDrive/checkpoints/ft_2_class/vit/{NOTEBOOK_STEM}_prt_ft_2class_ckpt"
    LOG_DIR = f"/content/drive/MyDrive/checkpoints/ft_2_class/vit/tb_logs/{NOTEBOOK_STEM}"
elif WHERE_AM_I == 'Local Linux':
    SAVE_DIR = f"/path/to/checkpoints/ft_2_class/vit/{NOTEBOOK_STEM}_prt_ft_2class_ckpt"
    LOG_DIR = f"/path/to/checkpoints/ft_2_class/vit/tb_logs/{NOTEBOOK_STEM}"
else:
    sys.exit("WHERE_AM_I is not set.")

TEST_RUN = False #True #False
TEST_RUN_TRAIN_STEP = 4
#TEST_RUN_VALID_STEP = 2

SCHEDULED_RUN = False #True #False
TARGET_HOUR = 21 #0 #23
TARGET_MIN = 30 #0

# モデル選択
MODEL_TYPE = "vit" #"deit" #"swinv2" #"swin" #"swinv2" #"swin" #"vit"  # "vit", "swin", "swinv2"を選択
# モデル名の定義
MODEL_NAMES = {
    "vit": "google/vit-base-patch16-384",
    "vitmae": "facebook/vit-mae-base",
    "swin": "microsoft/swin-base-patch4-window12-384-in22k",
    #"swinv2": "microsoft/swinv2-base-patch4-window8-256"   #batch_size 16でもんだいない
    "swinv2": "microsoft/swinv2-base-patch4-window16-256",  #batch_size 8でないとおそい
    #"swinv2": "microsoft/swinv2-base-patch4-window12to24-192to384-22kto1k-ft" #batch_sizse 2でないとおそい
    "deit": "facebook/deit-base-patch16-384", #batch_size 64はおそい
    "coatnet2": "timm/coatnet-2-rw_224.sw_in12k_ft_in1k",
    "gmlp": "timm/gmlp_s16_224.ra3_in1k"
}

IMAGE_SIZES = {
    "vit": 480,
    "vitmae": 480,
    "swin": 480,
    "swinv2": 480,
    "deit": 480,
    "coatnet2": 480
}

MODEL_NAME = MODEL_NAMES[MODEL_TYPE]
IMAGE_SIZE = IMAGE_SIZES[MODEL_TYPE]
BATCH_SIZE   = 64 #32 #64 #32 #16 #8 #16 #8 #2 #4 #8 #16
NUM_EPOCHS   = 15 #20 #50 #2 #200 #100 #2 #100 #2 #1 #25 #30 #2 #5            # 実運用では >10 が普通
LR_BASE      = 1e-5 #2e-5 #3e-5 #5e-5
SWA_EPOCH_RATIO = 0.75 #0.5 #0.75

USE_SAM = True  # SAMを使用するかどうか
SAM_RHO = 0.05  # SAMの探索半径

HIDDEN_DROP_OUT_PROB = 0.2 #0.4 #0.3 #0.2,
ATTENTION_DROP_OUT_PROB = 0.2 #0.4 #0.3 #0.2,

if WHERE_AM_I == 'Google Colab':
    root_dir = "/content/drive/MyDrive/data/ecg_dataset_480"
elif WHERE_AM_I == 'Local Linux':
    root_dir = "/path/to/data/ecg_dataset_480"
else:
    sys.exit("WHERE_AM_I is not set.")

train_dir, val_dir = [os.path.join(root_dir, x)
                                for x in ("train", "valid")]

# ラベル名の定義
label_names = ["tako", "AMI"]
label2id = {n:i for i,n in enumerate(label_names)}
id2label = {i:n for n,i in label2id.items()}
num_labels = len(label_names)

# non_tako_amiに含まれるディレクトリ名
non_tako_ami_dirs = ["nontako", "nonAMI", "normal"]

# データ拡張の設定
AUGMENTATION_CONFIG = {
    "randaugment": {
        "enabled": True,
        "n": 2,  # 適用する変換の数
        "m": 9   # 変換の強度 (0-10)
    },
    "random_erasing": {
        "enabled": True,
        "p": 0.5,  # 適用確率
        "scale": (0.02, 0.33),  # 消去領域のスケール
        "ratio": (0.3, 3.3)  # 消去領域のアスペクト比
    },
    "cutmix": {
        "enabled": True,
        "mixup_alpha": 0.3, #0.5, #0.7, #0.5, #0.3, #0.5, #0.3, #0.1, #0.4, #0.2, #1.0,
        "cutmix_alpha": 0.3, #0.5, #0.7, #0.5, #0.3, #0.5, #0.3, #0.1, #0.2, #1.0,
        "cutmix_minmax": None,
        "prob": 0.5, #0.05, #0.3, #0.5,
        "switch_prob": 0.5,
        "mode": 'batch',
        "label_smoothing": 0.25, #0.1
    }
}

# ---------------------------------------------------------
# 2. データセットラッパー
# ---------------------------------------------------------

# ECGなので上下左右のflipを外す
class RandAugmentNoFlip(RandAugment):
    def __init__(self, num_ops=2, magnitude=9, num_magnitude_bins=10, interpolation=None, fill=None):
        super().__init__(num_ops, magnitude, num_magnitude_bins, interpolation, fill)
        # ① 子モジュール一覧 self._modules を OrderedDict に取り出し
        modules = list(self._modules.items())
        # ② "Flip" を含むクラス名のモジュールだけ除外
        filtered = OrderedDict(
            (name, module)
            for name, module in modules
            if "Flip" not in module.__class__.__name__
        )
        # ③ 書き戻し
        self._modules = filtered



cfg = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    image_size=IMAGE_SIZE,
    hidden_dropout_prob=HIDDEN_DROP_OUT_PROB, #0.3, #0.2,
    attention_probs_dropout_prob=ATTENTION_DROP_OUT_PROB, #0.3, #0.2,
)

model = ViTForImageClassification.from_pretrained(
    FT3_CKPT_DIR,
    config=cfg,
    ignore_mismatched_sizes=True,
    local_files_only = True
)

processor = AutoImageProcessor.from_pretrained(FT3_CKPT_PARENT_DIR)
processor.save_pretrained(SAVE_DIR)

class PatientDataset(Dataset):
    def __init__(self, root_dir, is_train=True):
        self.samples = []
        self.is_train = is_train
        # 患者ディレクトリを走査
        for patient_dir in os.listdir(root_dir):
            patient_path = os.path.join(root_dir, patient_dir)
            if not os.path.isdir(patient_path):
                continue

            # ラベルディレクトリを走査
            for label_dir in os.listdir(patient_path):
                label_path = os.path.join(patient_path, label_dir)
                if not os.path.isdir(label_path):
                    continue

                # "tako"と"AMI"ディレクトリのみを対象
                if label_dir == "tako":
                    label = label2id["tako"]
                elif label_dir == "AMI":
                    label = label2id["AMI"]
                else:
                    continue

                # 画像ファイルを取得（PNG形式のみ）
                for img_file in os.listdir(label_path):
                    if img_file.lower().endswith('.png'):
                        img_path = os.path.join(label_path, img_file)
                        self.samples.append((img_path, label))

        # データ拡張の設定
        if is_train:
            self.transform = transforms.Compose([
                RandAugmentNoFlip(
                    num_ops=AUGMENTATION_CONFIG["randaugment"]["n"],
                    magnitude=AUGMENTATION_CONFIG["randaugment"]["m"],
                    interpolation=PIL.Image.BILINEAR,
                    fill=0,
                ),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                #transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])
            ])


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = PIL.Image.open(img_path).convert('RGB')

        if self.is_train:
            img = self.transform(img)
        else:
            img = self.transform(img)

        return {
            "pixel_values": img,
            "labels": torch.tensor(label)
        }

# データセットの作成
train_ds = PatientDataset(train_dir, is_train=True)
val_ds = PatientDataset(val_dir, is_train=False)

# ラベル分布を表示する関数
def print_label_distribution(dataset, dataset_name):
    labels = [label for _, label in dataset.samples]
    tako_count = labels.count(label2id["tako"])
    ami_count = labels.count(label2id["AMI"])
    total = len(labels)

    print(f"\n{dataset_name}のラベル分布:")
    print(f"tako: {tako_count} ({tako_count/total*100:.1f}%)")
    print(f"AMI: {ami_count} ({ami_count/total*100:.1f}%)")
    print(f"合計: {total}")

# 各データセットのラベル分布を表示
print_label_distribution(train_ds, "学習データ")
print_label_distribution(val_ds, "検証データ")


# CutMixの設定
mixup_fn = Mixup(
    mixup_alpha=AUGMENTATION_CONFIG["cutmix"]["mixup_alpha"],
    cutmix_alpha=AUGMENTATION_CONFIG["cutmix"]["cutmix_alpha"],
    cutmix_minmax=AUGMENTATION_CONFIG["cutmix"]["cutmix_minmax"],
    prob=AUGMENTATION_CONFIG["cutmix"]["prob"],
    switch_prob=AUGMENTATION_CONFIG["cutmix"]["switch_prob"],
    mode=AUGMENTATION_CONFIG["cutmix"]["mode"],
    label_smoothing=AUGMENTATION_CONFIG["cutmix"]["label_smoothing"],
    num_classes=num_labels
)


def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    labels       = torch.tensor([b["labels"] for b in batch])

    # 学習時のみCutMixを適用
    if batch[0]["pixel_values"].requires_grad:
        pixel_values, labels = mixup_fn(pixel_values, labels)

    return {
        "pixel_values": pixel_values,
        "labels": labels,
        "interpolate_pos_encoding": True
    }


# ---------------------------------------------------------
# 4. 評価指標（Accuracy だけ例示）
# ---------------------------------------------------------
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    # 混同行列（クラスごとの TP/FN/FP/TN を計算）
    cm = confusion_matrix(labels, preds)
    # cm[i,i] が TP_i,  cm.sum(axis=1)[i] - TP_i が FN_i
    # cm.sum(axis=0)[i] - TP_i が FP_i
    # TN_i = 全サンプル数 - TP_i - FN_i - FP_i
    TN = []
    FP = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i,:].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        TN.append(tn)
        FP.append(fp)
    specificity_per_class = [tn / (tn + fp) if (tn + fp) > 0 else 0.0
                             for tn, fp in zip(TN, FP)]
    specificity_macro = float(np.mean(specificity_per_class))

    # 既存の accuracy, precision, recall, f1…
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, labels=range(len(label_names))
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    metrics = {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision_macro": float(precision_macro),
        "recall_macro":    float(recall_macro),
        "f1_macro":        float(f1_macro),
        "specificity_macro": specificity_macro,    # ← ここで macro 特異度を追加
        "confusion_matrix": cm.tolist(),
    }

    for i, name in enumerate(label_names):
        metrics[f"precision_{name}"]   = float(precision[i])
        metrics[f"recall_{name}"]      = float(recall[i])
        metrics[f"f1_{name}"]          = float(f1[i])
        metrics[f"specificity_{name}"] = float(specificity_per_class[i])  # ← 各クラスの特異度

    return metrics


# ---------------------------------------------------------
# 5. Trainer で学習
# ---------------------------------------------------------
if TEST_RUN:
    args = TrainingArguments(
        output_dir               = SAVE_DIR,
        logging_dir              = LOG_DIR,
        report_to                = ["tensorboard"],
        logging_strategy          = "steps",
        logging_steps            = 50,        # 50 step ごとにログ
        eval_strategy            = "epoch",   # evaluation_strategy → eval_strategy
        save_strategy            = "epoch",   # 1 epoch ごとに保存
        learning_rate            = LR_BASE,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        weight_decay             = 1e-4,

        metric_for_best_model    = "eval_loss", #"accuracy", # ↑ で使う指標
        greater_is_better = False,
        dataloader_num_workers = 8, #4,  # データ読み込みの並列化
        dataloader_pin_memory = True,  # GPU転送の高速化
        label_smoothing_factor = 0.0, #0.1,
        #fp16 = True,  # NVIDIA GPU用のmixed precision
        # または
        bf16 = True,  # TPUや新しいNVIDIA GPU用のmixed precision
        max_steps = TEST_RUN_TRAIN_STEP,
    )
else:
    args = TrainingArguments(
        output_dir               = SAVE_DIR,
        logging_dir              = LOG_DIR,
        report_to                = ["tensorboard"],
        logging_strategy          = "epoch",
        #logging_steps            = 50,        # 50 step ごとにログ
        eval_strategy            = "epoch",   # evaluation_strategy → eval_strategy
        save_strategy            = "epoch",   # 1 epoch ごとに保存
        learning_rate            = LR_BASE,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        num_train_epochs         = NUM_EPOCHS,
        weight_decay             = 1e-4,
        load_best_model_at_end   = True,      # val 指標が最良の重みを最後に読み込む
        metric_for_best_model    = "eval_loss", #"accuracy", # ↑ で使う指標
        greater_is_better = False,
        dataloader_num_workers = 8, #4,  # データ読み込みの並列化
        dataloader_pin_memory = True,  # GPU転送の高速化
        label_smoothing_factor = 0.0, #0.1,
        #fp16 = True,  # NVIDIA GPU用のmixed precision
        # または
        bf16 = True,  # TPUや新しいNVIDIA GPU用のmixed precision
    )


class SWACallback(TrainerCallback):
    """
    Transformers Trainer で SAM + SWA を行うコールバック
    - epoch_ratio: SWA を開始するエポック (総エポックに対する割合)
    - swa_lr:     SWA 区間で使う LR (None なら base_lr / 10)
    """
    def __init__(self, epoch_ratio: float = 0.75, swa_lr: Optional[float] = None):
        self.epoch_ratio = epoch_ratio
        self.swa_lr = swa_lr

    # ---------- Trainer lifecycle hooks ----------
    def on_train_begin(self, args, state, control, **kwargs):
        model      = kwargs["model"]
        optimizer  = kwargs["optimizer"]

        self.swa_model = AveragedModel(model)               # (1) 平均用モデル
        base_lr        = optimizer.param_groups[0]["lr"]
        self.swa_scheduler = SWALR(                         # (2) SWA 用 LR スケジューラ
            optimizer,
            swa_lr = self.swa_lr or base_lr * 0.1
        )

        # SWA をいつ始めるか (グローバル step で持つ)
        total_updates = state.max_steps
        self.swa_start_step = int(total_updates * self.epoch_ratio)

    def on_step_end(self, args, state, control, **kwargs):
        # 学習後期なら SWA を更新
        if state.global_step >= self.swa_start_step:
            self.swa_model.update_parameters(kwargs["model"])
            self.swa_scheduler.step()

    def on_train_end(self, args, state, control, **kwargs):
        # model と dataloader は kwargs から取得
        model       = kwargs["model"]
        train_dl    = kwargs["train_dataloader"]

        # BatchNorm の統計を更新して averaged model を本モデルに反映
        update_bn(train_dl, self.swa_model, device=args.device)
        for swa_param, param in zip(self.swa_model.parameters(), model.parameters()):
            param.data.copy_(swa_param.data)


class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.
    α: クラス重み (リスト / 1-D Tensor でも可)
    γ: focusing parameter
    """
    def __init__(self, alpha=None, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is None:
            self.alpha = None
        else:
            # list → tensor, 正規化
            alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer("alpha", alpha / alpha.sum())

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        """targets は *整数ラベル* でも *one-hot* でも OK"""
        # ① softmax で確率へ
        probs = F.softmax(logits, dim=-1)        # (B,C)

        # ② one-hot に変換
        if targets.ndim == 1:                    # ラベルが整数のとき
            targets_onehot = F.one_hot(targets, num_classes=probs.size(1)).float()
        else:
            targets_onehot = targets.float()     # Mixup / CutMix で float になっている場合

        # ③ Focal Loss
        pt = (probs * targets_onehot).sum(dim=1)           # (B,)
        log_pt = pt.clamp(min=1e-7).log()                  # log(pt)
        focal_term = (1.0 - pt) ** self.gamma

        if self.alpha is not None:
            at = (self.alpha * targets_onehot).sum(dim=1)  # (B,)
            loss = -at * focal_term * log_pt
        else:
            loss = -focal_term * log_pt

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss

# ラベル分布をカウント
label_counts = Counter(label for _, label in train_ds.samples)
n_tako = label_counts[0]
n_ami = label_counts[1]
alpha = [1.0 / np.sqrt(n_tako), 1.0 / np.sqrt(n_ami)]    # 出現頻度の逆数の平方根
print("FocalLoss alpha =", alpha)
model.focal_loss = FocalLoss(
    alpha=alpha,
    gamma=2.0,
    reduction='mean',
)

class SAMTrainer(Trainer):
    def __init__(self, *args, sam_rho=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.sam_rho = sam_rho

    def compute_loss(self, model, inputs, return_outputs=False):
        labels  = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        loss = model.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        # ---------- 1st forward/backward ----------
        loss = self.compute_loss(model, inputs)
        self.accelerator.backward(loss)

        # ---------- SAM perturbation ----------
        eps_list = []                       # ← ここで保存
        with torch.no_grad():
            for p in model.parameters():
                if p.grad is None:
                    eps_list.append(None)
                    continue
                eps = self.sam_rho * p.grad / (p.grad.norm(p=2) + 1e-12)
                p.add_(eps)
                eps_list.append(eps)

        # ---------- 2nd forward/backward ----------
        second_loss = self.compute_loss(model, inputs)
        self.accelerator.backward(second_loss)

        # ---------- revert & average grads ----------
        with torch.no_grad():
            for p, eps in zip(model.parameters(), eps_list):
                if eps is None:
                    continue
                p.sub_(eps)        # 正しい eps を戻す
                p.grad.mul_(0.5)   # 勾配を平均

        return second_loss.detach()

# 学習履歴を保存するためのコールバックを追加します
class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.train_metrics = []
        self.eval_metrics = []
        self.lr_metrics = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if 'loss' in logs:
                self.train_metrics.append({
                    'step': state.global_step,
                    'loss': logs['loss']
                })
            if 'eval_loss' in logs:
                self.eval_metrics.append({
                    'step': state.global_step,
                    'loss': logs['eval_loss'],
                    'accuracy': logs['eval_accuracy']
                })
            if 'learning_rate' in logs:
                self.lr_metrics.append({
                    'step': state.global_step,
                    'lr': logs['learning_rate']
                })

def plot_training_history(trainer, metrics_callback):
    # 学習履歴の取得
    train_loss = [m['loss'] for m in metrics_callback.train_metrics]
    eval_loss = [m['loss'] for m in metrics_callback.eval_metrics]
    eval_acc = [m['accuracy'] for m in metrics_callback.eval_metrics]
    learning_rates = [m['lr'] for m in metrics_callback.lr_metrics]

    # プロットの作成
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

    # Lossのプロット
    ax1.plot(train_loss, label='Train Loss')
    ax1.plot(eval_loss, label='Validation Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracyのプロット
    ax2.plot(eval_acc, label='Validation Accuracy')
    ax2.set_title('Validation Accuracy')
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)

    # Learning Rateのプロット
    ax3.plot(learning_rates, label='Learning Rate')
    ax3.set_title('Learning Rate')
    ax3.set_xlabel('Steps')
    ax3.set_ylabel('Learning Rate')
    ax3.legend()
    ax3.grid(True)
    ax3.set_yscale('log')  # 対数スケールで表示

    plt.tight_layout()
    plt.show()

metrics_callback = MetricsCallback()

class BestModelCallback(TrainerCallback):
    def on_train_end(self, args, state, control, **kwargs):
        #best_model_path = os.path.join(args.output_dir, f"checkpoint-{state.best_model_checkpoint}")
        best_model_path = state.best_model_checkpoint
        print(f"\n最良のモデルは {best_model_path} に保存されています")
        print(f"最良の評価指標: {state.best_metric}")


# ────────────────
# ① オプティマイザをグループ化して定義
# ────────────────



if SCHEDULED_RUN: ek_schedule_exe.wait_until_target_time(TARGET_HOUR, TARGET_MIN)

# 1️⃣ Freeze backbone first
for p in model.vit.parameters():
    p.requires_grad = False

# ViT のバックボーン（self.model.vit）と分類ヘッド（self.model.classifier）を分けてパラメータ取得
backbone_params = list(model.vit.parameters())   # ★ 全パラメータを入れる
head_params     = list(model.classifier.parameters())
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 0.0},
    {"params": head_params,     "lr": LR_BASE * 2},
], weight_decay=1e-4)


# 2️⃣ Train head 3 epochs
head_args = dc.replace(
    args,
    num_train_epochs = 3,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.1,
    learning_rate = LR_BASE * 2,
    eval_strategy = "epoch",
)

head_trainer = SAMTrainer(
    model           = model,
    args            = head_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    data_collator   = collate_fn,
    optimizers      = (optimizer, None),
    callbacks       = [SWACallback(epoch_ratio=SWA_EPOCH_RATIO, swa_lr=LR_BASE*0.1)]
)
head_trainer.train()


# 3️⃣ Unfreeze all, lower LR, resume
for p in model.vit.parameters():
    p.requires_grad = True

backbone_params = [p for p in model.vit.parameters() if p.requires_grad]
head_params     = [p for p in model.classifier.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BASE * 0.05},
    {"params": head_params,     "lr": LR_BASE},
], weight_decay=1e-4)


full_args = dc.replace(
    args,
    num_train_epochs = NUM_EPOCHS,
    learning_rate = LR_BASE,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.1
)

# SAMの使用有無でTrainerを切り替え
if USE_SAM:
    full_trainer = SAMTrainer(
        model           = model,
        args            = full_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        data_collator   = collate_fn,
        optimizers      = (optimizer, None),
        callbacks       = [
            SWACallback(epoch_ratio=SWA_EPOCH_RATIO, swa_lr=LR_BASE*0.1),
            metrics_callback,
            BestModelCallback()
        ],
    )
else:
    full_trainer = Trainer(
        model           = model,
        args            = full_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        data_collator   = collate_fn,
        optimizers      = (optimizer, None),
        callbacks       = [
            SWACallback(epoch_ratio=SWA_EPOCH_RATIO, swa_lr=LR_BASE*0.1),
            metrics_callback,
            BestModelCallback()
        ],
    )

full_trainer.train(resume_from_checkpoint=head_trainer.state.best_model_checkpoint)
processor.save_pretrained(args.output_dir)
plot_training_history(full_trainer, metrics_callback)


print('Done. ', datetime.datetime.now(datetime.timezone(datetime.timedelta(hours = 9), 'JST')))